# Aula Prática 02 — Introdução aos Dados Tabulares

## 1. Tabela de brinquedo

Antes de carregar um dataset real, construo uma tabela pequena, à mão, para entender a gramática tabular sem a distração do volume de dados. Cada linha é uma observação, uma venda, e cada coluna é uma variável: produto, quantidade, preço, país.

In [1]:
import pandas as pd

sales = pd.DataFrame(
    {
        "product": ["Caneca", "Camiseta", "Caneca", "Adesivo", "Camiseta"],
        "quantity": [2, 1, 4, 10, 3],
        "unit_price": [35.0, 79.9, 35.0, 5.5, 79.9],
        "country": ["Brasil", "Brasil", "Portugal", "Brasil", "Argentina"],
    }
)

sales

,product,quantity,unit_price,country
0,Caneca,2,35.0,Brasil
1,Camiseta,1,79.9,Brasil
2,Caneca,4,35.0,Portugal
3,Adesivo,10,5.5,Brasil
4,Camiseta,3,79.9,Argentina


Dá pra classificar as variáveis dessa tabela sem muito esforço. `product` e `country` são categóricas nominais, rótulos sem ordem entre si. `quantity` é numérica discreta, uma contagem inteira de itens. `unit_price` é numérica contínua, pode assumir qualquer valor dentro de uma faixa. Essa classificação decide que operação faz sentido em cada coluna: tirar a média de `country` não significa nada, mas tirar a média de `unit_price` sim.

## 2. Três operações fundamentais

Toda interrogação de uma tabela nasce da combinação de três operações: selecionar uma coluna, derivar uma coluna nova e filtrar linhas.

In [2]:
sales["unit_price"]

0    35.0
1    79.9
2    35.0
3     5.5
4    79.9
Name: unit_price, dtype: float64

Selecionar uma coluna com `sales["unit_price"]` devolve uma Series, não um DataFrame: só o índice e os valores de `unit_price`, sem as outras colunas junto.

In [3]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales

,product,quantity,unit_price,country,revenue
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
2,Caneca,4,35.0,Portugal,140.0
3,Adesivo,10,5.5,Brasil,55.0
4,Camiseta,3,79.9,Argentina,239.7


`revenue` aparece como quinta coluna, igual a `quantity` vezes `unit_price` linha a linha. Na primeira linha, 2 canecas a 35 viram 70; confere com a conta de cabeça.

In [4]:
brazil_filter = sales["country"].eq("Brasil")
sales.loc[brazil_filter]

,product,quantity,unit_price,country,revenue
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
3,Adesivo,10,5.5,Brasil,55.0


O filtro devolve as 3 linhas em que `country` é "Brasil", com o índice original preservado (0, 1, 3). `.loc` filtra sem reordenar nem resetar o índice.

In [5]:
high_revenue_filter = sales["revenue"].gt(50)
sales.loc[brazil_filter & high_revenue_filter]

,product,quantity,unit_price,country,revenue
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
3,Adesivo,10,5.5,Brasil,55.0


As mesmas 3 linhas aparecem de novo: nessa tabela pequena, todas as vendas do Brasil já tinham `revenue` acima de 50, então o filtro combinado não eliminou nenhuma a mais aqui. Eliminaria se alguma venda do Brasil fosse barata.

Aviso importante: para combinar filtros booleanos em pandas eu uso `&` (e) e `|` (ou), nunca `and`/`or` do Python puro. Eles não sabem lidar com uma Series inteira de valores booleanos, só com um valor por vez.

## 3. Resumo estatístico

In [6]:
sales.describe().round(2)

,quantity,unit_price,revenue
count,5.00,5.00,5.00
mean,4.00,47.06,116.92
std,3.54,32.31,75.85
min,1.00,5.50,55.00
25%,2.00,35.00,70.00
50%,3.00,35.00,79.90
75%,4.00,79.90,140.00
max,10.00,79.90,239.70


Com só 5 linhas, o `describe()` serve mais pra conferir a conta do que pra tirar conclusão: `quantity` varia de 1 a 10, `unit_price` de 5,50 a 79,90 e `revenue` de 55 a 239,70, tudo dentro do que já dava pra ver olhando a tabela inteira.

## 4. Carregando o dataset real: Online Retail

Com a gramática fixada na tabela de brinquedo, carrego agora o dataset Online Retail (UCI Machine Learning Repository, id 352), versionado em `data/` na raiz do repositório. Renomeio as colunas para `snake_case` para manter a consistência com o resto do projeto. Como este notebook está em `Unidade_01/Aula_02/`, o caminho para os dados sobe duas pastas até a raiz.

In [7]:
from pathlib import Path
import pandas as pd

data_path = Path("..") / ".." / "data" / "online_retail.xlsx"

column_names = {
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
}

retail = pd.read_excel(data_path).rename(columns=column_names)

As variáveis do dataset seguem uma lógica parecida. `invoice_no`, `stock_code` e `customer_id` são categóricas nominais, funcionando como identificadores (mesmo `customer_id` sendo um número, ele não deve ter média tirada, é só um rótulo). `quantity` é numérica discreta, `unit_price` é numérica contínua, `invoice_date` é temporal, e `description` e `country` são categóricas nominais.

## 5. Ritual de inspeção inicial

Antes de qualquer pergunta de negócio, rodo os mesmos cinco passos em qualquer dataset novo: olhar os dados, medir o tamanho, quantificar ausentes, resumir os números e contar categorias.

Primeiro, olho os dados reais.

In [8]:
retail.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


As 5 primeiras linhas são todas da mesma fatura (536365), o que já mostra que uma fatura normalmente tem vários produtos, uma linha por item.

In [9]:
retail.tail()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


As últimas linhas são de dezembro de 2011, da fatura 581587, na França. Isso confirma que a base está ordenada do primeiro ao último pedido, não embaralhada.

Agora meço tamanho e tipos.

In [10]:
retail.shape

(541909, 8)

541.909 linhas e 8 colunas. Confiro os tipos de cada uma na próxima célula.

In [11]:
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   invoice_no    541909 non-null  object        
 1   stock_code    541909 non-null  object        
 2   description   540455 non-null  object        
 3   quantity      541909 non-null  int64         
 4   invoice_date  541909 non-null  datetime64[us]
 5   unit_price    541909 non-null  float64       
 6   customer_id   406829 non-null  float64       
 7   country       541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


O dataset tem 541.909 linhas e 8 colunas. Como esperado, `customer_id` aparece como `float64`, não `int64`. É sinal de que a coluna tem valores ausentes: o pandas precisa de ponto flutuante para representar `NaN` numérico.

Em seguida, quantifico os ausentes.

In [12]:
retail.isna().sum().sort_values(ascending=False)

customer_id     135080
description       1454
stock_code           0
invoice_no           0
quantity             0
invoice_date         0
unit_price           0
country              0
dtype: int64

`customer_id` tem 135.080 valores ausentes e `description` tem 1.454. As demais colunas não têm nenhum ausente. Quase um quarto das linhas não tem cliente identificado, o que já é um sinal de que nem toda análise que depende de `customer_id` vai poder usar o dataset inteiro.

Resumo as variáveis numéricas.

In [13]:
retail.describe()

,quantity,invoice_date,unit_price,customer_id
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


Duas coisas chamam atenção no describe. `quantity` tem mínimo de -80.995 e máximo de 80.995, e quantidade negativa não existe de verdade, então ou são cancelamentos registrados como negativo ou erro de digitação. `unit_price` tem mínimo de -11.062,06, e preço negativo é ainda mais estranho que quantidade negativa. Por enquanto só registro o estranhamento, nenhum valor foi alterado ainda.

Por fim, conto as categorias.

In [14]:
retail["country"].value_counts().head(10)

country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

In [15]:
retail["country"].nunique()

38

O dataset cobre 38 países, mas o Reino Unido domina com 495.478 das 541.909 linhas, mais de 90%. Qualquer conclusão sobre "o perfil de compra típico" neste dataset é, na prática, uma conclusão sobre o Reino Unido. Os outros 37 países juntos são uma fatia pequena.

## 6. Interrogação de dados: seleção e filtragem

In [16]:
germany_filter = retail["country"].eq("Germany")
germany_sales = retail.loc[germany_filter].copy()
germany_sales.shape

(9495, 8)

A Alemanha sozinha tem 9.495 linhas, bem menos que o Reino Unido, mas ainda assim o segundo país com mais linhas no `value_counts` de antes.

In [17]:
expensive_outside_uk_filter = (
    retail["unit_price"].gt(100)
    & retail["country"].ne("United Kingdom")
)

result_columns = ["invoice_no", "description", "unit_price", "country"]
expensive_outside_uk = retail.loc[
    expensive_outside_uk_filter,
    result_columns,
].copy()

expensive_outside_uk.head()

,invoice_no,description,unit_price,country
39777,539688,CARRIAGE,150.0,France
45622,C540271,Manual,1126.0,Spain
54382,540946,VINTAGE BLUE KITCHEN CABINET,125.0,Cyprus
54395,540946,VINTAGE RED KITCHEN CABINET,125.0,Cyprus
70795,542102,VINTAGE RED KITCHEN CABINET,125.0,Singapore


`.loc[linhas, colunas]` seleciona as linhas por condição booleana e as colunas por nome na mesma chamada. É a combinação de filtro e seleção em uma única operação.

## 7. Perguntas para consolidar

**Pergunta 1: as transações do Brasil.** *Quantas linhas têm `country == "Brazil"`?*

In [18]:
# Filtro: quantas linhas têm country == "Brazil"
brazil_filter = retail["country"].eq("Brazil")
result_1 = retail.loc[brazil_filter].shape[0]
print(f"Transações do Brasil: {result_1}")

Transações do Brasil: 32


O roteiro da aula previa que o Brasil não apareceria no dataset, ou teria contagem zero. Na minha execução ele aparece com 32 transações. O roteiro errou a expectativa, mas o jeito certo de descobrir isso é rodando o filtro, não assumindo o resultado.

**Pergunta 2: quantidades negativas.** *Quantas linhas têm `quantity < 0`?*

In [19]:
# Diagnóstico: quantas linhas têm quantity < 0
negative_qty_filter = retail["quantity"].lt(0)
negative_qty_data = retail.loc[negative_qty_filter]
print(f"Linhas com quantidade negativa: {len(negative_qty_data)}")

negative_qty_data.head()

Linhas com quantidade negativa: 10624


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


10.624 linhas têm quantidade negativa. Olhando a amostra, os `invoice_no` dessas linhas começam com "C", um padrão que investigo com mais detalhe mais adiante, mas que já sugere cancelamento de pedido, não erro de digitação.

**Pergunta 3: preço zero e cliente ausente.** *Quantas linhas têm `unit_price == 0` e `customer_id` ausente ao mesmo tempo?*

In [20]:
# Filtro combinado
zero_price_filter = retail["unit_price"].eq(0)
customer_na_filter = retail["customer_id"].isna()

result_3 = retail.loc[zero_price_filter & customer_na_filter]
print(f"Linhas com preço=0 E customer_id ausente: {len(result_3)}")

Linhas com preço=0 E customer_id ausente: 2475


2.475 das 2.515 linhas com preço zero também não têm cliente identificado, quase 98%. Preço zero não parece ser desconto genuíno, e sim sintoma do mesmo problema: lançamento administrativo ou ajuste de estoque que nunca teve cliente nem preço de verdade associado.

## 8. Diagnóstico de qualidade (sem tratamento)

Nesta fase só documento os problemas encontrados. Nenhum valor é corrigido, removido ou substituído agora, decidir o que fazer com cada um é uma etapa futura, separada do diagnóstico.

Quantidades negativas, de novo, agora com mais detalhe.

In [21]:
negative_quantity_filter = retail["quantity"].lt(0)
negative_quantity = retail.loc[negative_quantity_filter].copy()

print(f"{len(negative_quantity):,} linhas com quantidade negativa")

negative_quantity.head()

10,624 linhas com quantidade negativa


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [22]:
# Análise do padrão
negative_invoices = retail.loc[negative_quantity_filter, "invoice_no"]
starts_with_c = negative_invoices.str.startswith("C", na=False)

print(f"Começam com C: {starts_with_c.sum():,}")
print(f"Total de negativas: {len(negative_invoices):,}")

Começam com C: 9,288


Total de negativas: 10,624


9.288 das 10.624 linhas negativas, 87%, têm `invoice_no` começando com "C". Bate exatamente com o número que o roteiro previa, e reforça a hipótese de cancelamento: o "C" provavelmente é um prefixo que o sistema de vendas usa para marcar fatura de cancelamento ou estorno. Os 13% restantes, sem o "C", ainda são um ponto em aberto.

Preços zerados.

In [23]:
zero_price_filter = retail["unit_price"].eq(0)
zero_price = retail.loc[zero_price_filter].copy()

print(f"{len(zero_price):,} linhas com preço zero")

zero_price.loc[
    :, ["invoice_no", "description", "quantity", "customer_id"]
].head()

2,515 linhas com preço zero


,invoice_no,description,quantity,customer_id
622,536414,NaN,56,NaN
1970,536545,NaN,1,NaN
1971,536546,NaN,1,NaN
1972,536547,NaN,1,NaN
1987,536549,NaN,1,NaN


2.515 linhas têm preço zero, e a amostra confirma o que a Pergunta 3 já indicava: muitas dessas linhas também têm `description` e `customer_id` ausentes. O padrão se repete, parece ser uma categoria inteira de registro, ajuste ou perda de estoque, que convive no mesmo dataset que as vendas de verdade.

Duplicatas.

In [24]:
duplicate_count = retail.duplicated().sum()
print(f"{duplicate_count:,} linhas duplicadas")

5,268 linhas duplicadas


5.268 linhas são duplicatas exatas, todas as 8 colunas idênticas. Sem saber a origem, reenvio do sistema ou erro de exportação, não dá pra simplesmente assumir que são erro e descartar, mas também não parecem ser 5.268 vendas genuinamente repetidas no mesmo segundo. Fica registrado como mais um problema para decidir o tratamento depois.

## 9. Para casa: perguntas, ficha e exploração livre

As três perguntas de negócio (transações do Brasil, quantidades negativas, preço zero e cliente ausente) já foram respondidas com código comentado e interpretação na seção 7, durante a aula. Não repito aqui pra não duplicar o mesmo resultado.

Ficha de reconhecimento do dataset: 541.909 linhas e 8 colunas. `invoice_no`, `stock_code` e `customer_id` são categóricas nominais e funcionam como identificadores, `quantity` é numérica discreta, `unit_price` é numérica contínua, `invoice_date` é temporal, e `description` e `country` são categóricas nominais. `customer_id` tem 135.080 valores ausentes, um quarto das linhas, e `description` tem 1.454; as outras 6 colunas não têm nenhum ausente. Três problemas ficaram diagnosticados: 10.624 linhas com quantidade negativa, das quais 9.288 (87%) têm `invoice_no` começando com "C", provável cancelamento; 2.515 linhas com preço zero, das quais 2.475 (98%) também não têm `customer_id`, provável lançamento administrativo e não venda real; e 5.268 linhas inteiramente duplicadas, de origem ainda não explicada.

Escolhi explorar quanto do volume de transações é de produtos relacionados a Natal (descrição contendo "CHRISTMAS"), e quais países mais compram esses itens. Uso só filtro e seleção, sem agrupamento.

In [25]:
christmas_filter = retail["description"].str.contains("CHRISTMAS", na=False)
christmas_sales = retail.loc[christmas_filter].copy()

print(f"Transações com 'CHRISTMAS' na descrição: {len(christmas_sales):,}")
christmas_sales["country"].value_counts().head(5)

Transações com 'CHRISTMAS' na descrição: 25,132


country
United Kingdom    23601
EIRE                430
Germany             280
France              226
Spain               120
Name: count, dtype: int64

25.132 das 541.909 linhas, cerca de 4,6%, mencionam "CHRISTMAS" na descrição, um volume relevante pra um único tema de produto. O Reino Unido concentra 23.601 dessas transações, 94%, acima da sua participação geral no dataset, 91%. Itens de Natal parecem proporcionalmente ainda mais concentrados no mercado doméstico do que o restante do catálogo.

## Conclusão

O ritual de inspeção (`head`, `shape`, `info`, `describe`, `value_counts`, `isna`) não corrigiu nada, só revelou onde estão os problemas: quantidades e preços negativos, preços zerados quase sempre sem cliente, e milhares de linhas duplicadas. A regra que fica da aula é que diagnosticar e tratar são etapas separadas, e tratar antes de diagnosticar é decidir no escuro.

> Observação: o arquivo `data/online_retail.xlsx` está versionado na raiz deste repositório, justamente para quem clonar não precisar baixar o dataset de novo.